# FilmDB — 3-Stage RAG Evaluation
> **Run on Kaggle GPU.** Tests all 21 queries across 3 pipeline modes to validate embedding quality and measure the value-add of reranking and LLM curation.
>
> **Kaggle Datasets needed:**
> - `curated_corpus.parquet` (your FilmDB Curated Corpus dataset)
> - `embeddings.npy` + `embedding_ids.txt` (your FilmDB Embeddings dataset)
>
> **Kaggle Secret needed:** `GROQ_API_KEY` (for Mode C — LLM curation)

## 1. Install Dependencies

In [1]:
# sentence-transformers natively supports BGE rerankers
!pip install -q sentence-transformers groq
print('Dependencies ready!')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.1 MB/s eta 0:00:00
Dependencies ready!


## 2. Load Corpus, Embeddings & Model

In [2]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from sentence_transformers import SentenceTransformer, CrossEncoder
import numpy as np
import pandas as pd
from pathlib import Path
import torch

# ── Paths (adjust dataset slugs to match your Kaggle datasets) ──────────
CORPUS_PATH = Path('/kaggle/input/datasets/naren2308/filmdb-recommendation-engine/curated_corpus.parquet')
EMB_PATH    = Path('/kaggle/input/datasets/naren2308/filmdb-recommendation-engine/embeddings.npy')
IDS_PATH    = Path('/kaggle/input/datasets/naren2308/filmdb-recommendation-engine/embedding_ids.txt')
OUT_PATH    = Path('/kaggle/working/')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

print('Loading corpus...')
df = pd.read_parquet(CORPUS_PATH)
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['imdb_votes'] = pd.to_numeric(df['imdb_votes'], errors='coerce')
# ── Genre sanity check: keep only movie-type entries ─────────────────────
# Drop video games, TV episodes, etc. that may have leaked into the corpus
MOVIE_TYPES = {'movie', 'short', 'tvMovie'}
if 'title_type' in df.columns:
    pre = len(df)
    df = df[df['title_type'].isin(MOVIE_TYPES)].reset_index(drop=True)
    print(f'  Genre filter: {pre:,} → {len(df):,} entries (removed {pre - len(df):,} non-movie entries)')
else:
    print('  No title_type column found — skipping genre sanity filter')
print(f'  Corpus size after filtering: {len(df):,}')

print('Loading embeddings...')
embeddings = np.load(EMB_PATH)
ids        = IDS_PATH.read_text().splitlines()
print(f'  Embeddings shape: {embeddings.shape}')

# Build fast lookup maps
id_to_idx     = {imdb_id: i for i, imdb_id in enumerate(ids)}
df_indexed    = df.set_index('imdb_id')

print('Loading BGE bi-encoder...')
bi_encoder = SentenceTransformer('BAAI/bge-base-en-v1.5', device=device)

print('Loading BGE cross-encoder reranker (forcing cuda:0)...')
reranker = CrossEncoder('BAAI/bge-reranker-large', max_length=512, device='cuda:0', model_kwargs={'torch_dtype': torch.float16})

print('All models loaded!')

Device: cuda
Loading corpus...
  No title_type column found — skipping genre sanity filter
  Corpus size after filtering: 98,272
Loading embeddings...
  Embeddings shape: (98272, 768)
Loading BGE bi-encoder...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

All models loaded!


## 3. Query Definitions
All 21 queries from `all_queries.txt`, organized by strategy with their filter configs.

In [3]:
QUERIES = [
    # S1: Narrative / Semantic Plot Search
    {'id': 'Q01', 'strategy': 'S1_Narrative', 'filter': {},
     'text': 'Recommend me a mind-bending movie about dreams and shifting reality.'},
    {'id': 'Q02', 'strategy': 'S1_Narrative', 'filter': {},
     'text': 'Recommend me a quiet, slice-of-life story about a family in Tokyo.'},
    {'id': 'Q03', 'strategy': 'S1_Narrative', 'filter': {},
     'text': 'Recommend me a gritty crime epic about the rise and fall of a mob boss.'},

    # S2: Prestige (Oscar)
    {'id': 'Q04', 'strategy': 'S2_Prestige', 'filter': {'oscar_wins': (1, None)},
     'text': 'Recommend me an Oscar-winning film that is a sweeping historical epic.'},
    {'id': 'Q05', 'strategy': 'S2_Prestige', 'filter': {'oscar_wins': (0, 0)},
     'text': 'Recommend me a critically acclaimed classic film that was nominated for major awards but controversially never won.'},
    {'id': 'Q06', 'strategy': 'S2_Prestige', 'filter': {'oscar_wins': (1, None)},
     'text': 'Recommend me an Oscar-winning film that is a beautiful musical about jazz and passion.'},

    # S3: Cultural / Regional Cinema
    {'id': 'Q07', 'strategy': 'S3_Cultural', 'filter': {'original_language': 'ja'},
     'text': 'Recommend me a Japanese dark and violent revenge thriller.'},
    {'id': 'Q08', 'strategy': 'S3_Cultural', 'filter': {'original_language': ('hi', 'ta', 'te', 'ml', 'kn', 'bn')},
     'text': 'Recommend me a gritty Indian gangster epic, exploring the criminal underworld, police encounters, and street violence.'},
    {'id': 'Q09', 'strategy': 'S3_Cultural', 'filter': {'original_language': 'es'},
     'text': 'Recommend me a gripping Spanish-language crime thriller.'},

    # S4: Platform Availability
    {'id': 'Q10', 'strategy': 'S4_Platform', 'filter': {'netflix': 1},
     'text': 'Recommend me a fast-paced action movie with great set pieces currently streaming on Netflix.'},
    {'id': 'Q11', 'strategy': 'S4_Platform', 'filter': {'prime_video': 1},
     'text': 'Recommend me a scary supernatural horror film currently streaming on Amazon Prime Video.'},
    {'id': 'Q12', 'strategy': 'S4_Platform', 'filter': {'disney_plus': 1},
     'text': 'Recommend me a laugh-out-loud feel-good comedy currently streaming on Disney+.'},

    # S5: Film Similarity (semantic "movies like X")
    {'id': 'Q13', 'strategy': 'S5_Similarity', 'filter': {},
     'text': 'Recommend me movies similar to The Matrix: philosophical sci-fi, simulated reality, mind-bending action.'},
    {'id': 'Q14', 'strategy': 'S5_Similarity', 'filter': {},
     'text': 'Recommend me movies similar to Boogie Nights: epic ensemble casts, rise-and-fall narratives, kinetic energy, 1970s/80s excess.'},

    # S6: Director Spotlight
    {'id': 'Q15', 'strategy': 'S6_Director', 'filter': {'director': 'Stanley Kubrick'},
     'text': 'What are the best Stanley Kubrick films, from his greatest masterpieces to essential watches?'},
    {'id': 'Q16', 'strategy': 'S6_Director', 'filter': {'director': 'Satyajit Ray'},
     'text': 'What are the best Satyajit Ray films to watch and in what order?'},

    # S7: Diversity (MMR-based)
    {'id': 'Q17', 'strategy': 'S7_Diversity', 'filter': {},
     'text': 'Recommend me a highly diverse list of space exploration sci-fi movies across different eras and styles.'},

    # S8: Mood / Vibe
    {'id': 'Q18', 'strategy': 'S8_Mood', 'filter': {},
     'text': 'Recommend me a movie with this exact vibe: cozy, rainy-day, low-stakes, and warm.'},
    {'id': 'Q19', 'strategy': 'S8_Mood', 'filter': {},
     'text': 'Recommend me a movie with this exact vibe: dark, atmospheric, slow-burn dread.'},

    # S9: Hidden Gems
    {'id': 'Q20', 'strategy': 'S9_HiddenGem', 'filter': {'max_votes': 10000, 'min_rating': 7.0},
     'text': 'Recommend me an absolute masterpiece of cinema that no one talks about.'},
    {'id': 'Q21', 'strategy': 'S9_HiddenGem', 'filter': {'max_votes': 10000, 'min_rating': 7.0},
     'text': 'Recommend me a highly rated psychological thriller that went completely unnoticed by mainstream audiences.'},
]

print(f'Loaded {len(QUERIES)} test queries across 9 strategies.')

Loaded 21 test queries across 9 strategies.


## 4. Retrieval Functions

In [4]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

TOP_K_DENSE   = 50   # candidates for reranker
TOP_K_FINAL   = 10   # final results shown

def apply_filter(filter_cfg: dict) -> np.ndarray:
    """Returns boolean mask of rows that pass the filter."""
    # ── Global quality guard: exclude incomplete / fake-rated entries ──────
    mask = (
        df['year'].notna() & (df['year'] > 0) &
        df['imdb_votes'].notna() & (df['imdb_votes'] > 0)
    ).values
    if 'oscar_wins' in filter_cfg:
        lo, hi = filter_cfg['oscar_wins']
        if lo is not None: mask &= (df['oscar_wins'].fillna(0) >= lo).values
        if hi is not None: mask &= (df['oscar_wins'].fillna(0) <= hi).values
    if 'original_language' in filter_cfg:
        lang = filter_cfg['original_language']
        if isinstance(lang, tuple):
            mask &= df['original_language'].isin(lang).values
        else:
            mask &= (df['original_language'] == lang).values
    if 'director' in filter_cfg:
        mask &= (df['director'].fillna('').str.contains(filter_cfg['director'], case=False, na=False)).values
    if 'netflix' in filter_cfg:
        mask &= (df['netflix'].fillna(0) == 1).values
    if 'prime_video' in filter_cfg:
        mask &= (df['prime_video'].fillna(0) == 1).values
    if 'disney_plus' in filter_cfg:
        mask &= (df['disney_plus'].fillna(0) == 1).values
    if 'hulu' in filter_cfg:
        mask &= (df['hulu'].fillna(0) == 1).values
    if 'max_votes' in filter_cfg:
        # Hidden gem: must have between MIN_GEM_VOTES and max_votes
        MIN_GEM_VOTES = 100  # exclude fake-rated micro-releases
        mask &= (df['imdb_votes'].fillna(0) >= MIN_GEM_VOTES).values
        mask &= (df['imdb_votes'].fillna(99999) <= filter_cfg['max_votes']).values
    if 'min_rating' in filter_cfg:
        # Ensure hidden gems are genuinely well-rated (eliminates video games, noise)
        mask &= (pd.to_numeric(df['imdb_rating'], errors='coerce').fillna(0) >= filter_cfg['min_rating']).values
    return mask


def mmr_rerank(query_vec, cand_vecs, cand_indices, top_k=10, lambda_=0.6):
    """Maximal Marginal Relevance for diversity."""
    selected, remaining = [], list(range(len(cand_indices)))
    while len(selected) < top_k and remaining:
        scores = []
        for i in remaining:
            rel   = cos_sim([query_vec], [cand_vecs[i]])[0][0]
            div   = max([cos_sim([cand_vecs[i]], [cand_vecs[s]])[0][0] for s in selected], default=0)
            scores.append(lambda_ * rel - (1 - lambda_) * div)
        best = remaining[np.argmax(scores)]
        selected.append(best)
        remaining.remove(best)
    return [cand_indices[i] for i in selected]


def mode_a_dense(query_text, filter_cfg, strategy, top_k=TOP_K_FINAL):
    """Stage 1 only: cosine similarity with optional metadata pre-filter."""
    q_vec = bi_encoder.encode([query_text], normalize_embeddings=True)[0]
    mask  = apply_filter(filter_cfg)
    valid_indices = np.where(mask)[0]
    if len(valid_indices) == 0:
        return []
    sims = cos_sim([q_vec], embeddings[valid_indices])[0]
    # For Hidden Gem: blend gem_score with semantic similarity
    if strategy == 'S9_HiddenGem':
        gem_scores = df['gem_score'].fillna(0).values[valid_indices]
        # Normalize both to [0,1] then blend: 60% gem quality, 40% semantic relevance
        gem_norm  = (gem_scores - gem_scores.min()) / (gem_scores.max() - gem_scores.min() + 1e-9)
        sim_norm  = (sims - sims.min()) / (sims.max() - sims.min() + 1e-9)
        blend     = 0.6 * gem_norm + 0.4 * sim_norm
        top_local = np.argsort(blend)[::-1][:top_k]
    elif strategy == 'S7_Diversity':
        cand_local = np.argsort(sims)[::-1][:TOP_K_DENSE]
        cand_vecs  = embeddings[valid_indices[cand_local]]
        mmr_local  = mmr_rerank(q_vec, cand_vecs, list(range(len(cand_local))), top_k)
        top_local  = cand_local[mmr_local]
    else:
        top_local = np.argsort(sims)[::-1][:top_k]
    return [(valid_indices[i], float(sims[i])) for i in top_local]


def mode_b_rerank(query_text, filter_cfg, strategy):
    """Stage 1 + 2: dense retrieval then cross-encoder reranking."""
    # Stage 1: Get top-50 candidates
    q_vec = bi_encoder.encode([query_text], normalize_embeddings=True)[0]
    mask  = apply_filter(filter_cfg)
    valid_indices = np.where(mask)[0]
    if len(valid_indices) == 0:
        return []
    sims      = cos_sim([q_vec], embeddings[valid_indices])[0]
    top_local = np.argsort(sims)[::-1][:TOP_K_DENSE]
    cand_indices = [valid_indices[i] for i in top_local]

    # Stage 2: Cross-encoder reranking (raw logit scores for wider discriminative range)
    emb_texts = df['embedding_text'].iloc[cand_indices].tolist()
    pairs = [[query_text, t[:512]] for t in emb_texts]
    raw_scores = reranker.predict(pairs)  # raw logits, NOT sigmoid-normalized
    reranked = sorted(zip(cand_indices, raw_scores.tolist()), key=lambda x: x[1], reverse=True)[:TOP_K_FINAL]
    return reranked  # list of (df_index, raw_logit_score)


print('Retrieval functions ready!')

Retrieval functions ready!


## 5. LLM Curator (Groq)

In [5]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

# Read your Groq API key from Kaggle Secrets
# Go to: Notebook Settings → Add-ons → Secrets → Add GROQ_API_KEY
try:
    secrets = UserSecretsClient()
    groq_key = secrets.get_secret('GROQ_API_KEY')
    groq_client = Groq(api_key=groq_key)
    LLM_ENABLED = True
    print('Groq client ready (Mode C enabled)')
except Exception as e:
    print(f'Groq key not found: {e} — Mode C will be skipped')
    LLM_ENABLED = False

GROQ_MODEL = 'llama-3.3-70b-versatile'

def mode_c_llm(query_text, reranked_results):
    """Stage 3: LLM curation using top-10 reranked results."""
    if not LLM_ENABLED or not reranked_results:
        return '_LLM not available or no results_'

    # Build context block from top-10 results
    lines = []
    for idx, score in reranked_results:
        row = df.iloc[idx]
        year_val = row.get('year')
        year = int(float(year_val)) if pd.notna(year_val) else 0
        lines.append(
            f"- **{row.get('title', 'Unknown')} ({year})** | "
            f"Rating: {row.get('imdb_rating', 'N/A')} | "
            f"Director: {row.get('director', 'Unknown')} | "
            f"Genres: {row.get('genres', 'Unknown')}"
        )
    context = '\n'.join(lines)

    prompt = f"""You are a senior film curator at a world-class cinematheque. A viewer has submitted the following request:

REQUEST: \"{query_text}\"

The semantic retrieval pipeline has surfaced the following candidate films:
{context}

Your task:
1. Select EXACTLY 5 films from the candidates above that best satisfy the request.
2. For each film, provide: its title, year, and a 2-3 sentence explanation of WHY it fits the request.
3. Note any relevant caveats (language, tone, era, content warnings) where appropriate.
4. Do NOT recommend films outside the candidate list.
5. Format your response as a clean markdown table with columns: Film | Why it fits | Caveats.

Be precise, authoritative, and genuinely useful. Your response must contain all 5 picks, fully described."""

    response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        max_tokens=900,
        temperature=0.3,
    )
    return response.choices[0].message.content


print('LLM curator ready!')

Groq client ready (Mode C enabled)
LLM curator ready!


## 6. Run Evaluation — All 21 Queries
**Checkpointing enabled.** If this cell crashes, just run it again. It will skip queries it has already saved to the report file.

In [6]:
import time
import os
import json
import shutil
from pathlib import Path

report_file = OUT_PATH / 'rag_evaluation_results.md'
json_file   = OUT_PATH / 'showcase_results.json'
completed_queries = set()
all_results = []

if not report_file.exists():
    # Look for an uploaded checkpoint in Kaggle datasets
    checkpoint_input = list(Path('/kaggle/input').rglob('rag_evaluation_results*.md'))
    if checkpoint_input:
        print(f'Found uploaded checkpoint: {checkpoint_input[0]}')
        shutil.copy(checkpoint_input[0], report_file)
        print('Copied checkpoint to working directory.')

if report_file.exists():
    with open(report_file, 'r', encoding='utf-8') as f:
        existing_content = f.read()
        for q in QUERIES:
            if f"## {q['id']}:" in existing_content:
                completed_queries.add(q['id'])
    print(f"Resuming from checkpoint. {len(completed_queries)} queries already completed.")
else:
    with open(report_file, 'w', encoding='utf-8') as f:
        f.write('# FilmDB RAG Evaluation Report\n')
        f.write(f'> Corpus: 99k movies | Bi-encoder: BGE-base | Reranker: BGE-reranker-large | LLM: {GROQ_MODEL}\n')
        f.write('---\n')

def fmt_row(df_idx, score, mode='dense'):
    row = df.iloc[df_idx]
    title   = str(row.get('title', 'Unknown'))[:40]
    year_val = row.get('year')
    year    = int(float(year_val)) if pd.notna(year_val) else 0
    rating  = row.get('imdb_rating', 'N/A')
    votes_val = row.get('imdb_votes')
    votes   = int(float(votes_val)) if pd.notna(votes_val) else 0
    if mode == 'rerank':
        return f'| {title} | {year} | {rating} | {votes:,} | {score:.3f} |'
    return f'| {title} | {year} | {rating} | {votes:,} | {score:.3f} |'

for q in QUERIES:
    if q['id'] in completed_queries:
        print(f"Skipping {q['id']} (already completed)")
        continue

    print(f"\n--- Running {q['id']}: {q['text'][:60]} ---")
    t0 = time.time()

    # Mode A: Dense only
    print("  [Stage 1] Running Dense Retrieval...")
    mode_a = mode_a_dense(q['text'], q['filter'], q['strategy'])

    # Mode B: Dense + Rerank
    print("  [Stage 2] Running Cross-Encoder Rerank...")
    mode_b = mode_b_rerank(q['text'], q['filter'], q['strategy'])

    # Mode C: Full pipeline
    print("  [Stage 3] Running LLM Curation (Groq)...\n")
    mode_c_text = mode_c_llm(q['text'], mode_b)

    elapsed = time.time() - t0

    # Build markdown section
    q_lines = [
        f"## {q['id']}: {q['text']}\n",
        f"**Strategy:** `{q['strategy']}` | **Time:** {elapsed:.1f}s\n",
        '\n',
        '### Mode A — Dense Only (cosine similarity)\n',
        '| Title | Year | IMDB | Votes | Sim Score |\n',
        '|---|---|---|---|---|\n',
    ]
    for idx, score in mode_a:
        q_lines.append(fmt_row(idx, score, 'dense') + '\n')

    q_lines += [
        '\n',
        '### Mode B — Dense + Cross-Encoder Rerank\n',
        '| Title | Year | IMDB | Votes | Rerank Score |\n',
        '|---|---|---|---|---|\n',
    ]
    for idx, score in mode_b:
        q_lines.append(fmt_row(idx, score, 'rerank') + '\n')

    q_lines += [
        '\n',
        '### Mode C — Full Pipeline (LLM Curated)\n',
        f'{mode_c_text}\n',
        '\n---\n',
    ]

    # Accumulate structured results for JSON export (full metadata inline)
    def to_record(idx, score):
        r = df.iloc[idx]
        yv = r.get('year'); vv = r.get('imdb_votes')
        return {
            'title':  str(r.get('title', 'Unknown')),
            'year':   int(float(yv)) if pd.notna(yv) else 0,
            'rating': r.get('imdb_rating', None),
            'votes':  int(float(vv)) if pd.notna(vv) else 0,
            'score':  round(score, 4),
        }
    all_results.append({
        'id':      q['id'],
        'query':   q['text'],
        'strategy': q['strategy'],
        'elapsed': round(elapsed, 2),
        'mode_a':  [to_record(idx, s) for idx, s in mode_a],
        'mode_b':  [to_record(idx, s) for idx, s in mode_b],
        'mode_c':  mode_c_text,
    })

    with open(report_file, 'a', encoding='utf-8') as f:
        f.writelines(q_lines)

    print(f'  Done in {elapsed:.1f}s')

with open(json_file, 'w', encoding='utf-8') as f:
    json.dump(all_results, f)

print('\nAll 21 queries evaluated!')

Found uploaded checkpoint: /kaggle/input/datasets/naren2308/filmdb-recommendation-engine/rag_evaluation_results (5).md
Copied checkpoint to working directory.
Resuming from checkpoint. 20 queries already completed.
Skipping Q01 (already completed)
Skipping Q02 (already completed)
Skipping Q03 (already completed)
Skipping Q04 (already completed)
Skipping Q05 (already completed)
Skipping Q06 (already completed)
Skipping Q07 (already completed)

--- Running Q08: Recommend me a gritty Indian gangster epic, exploring the cr ---
  [Stage 1] Running Dense Retrieval...
  [Stage 2] Running Cross-Encoder Rerank...
  [Stage 3] Running LLM Curation (Groq)...

  Done in 3.6s
Skipping Q09 (already completed)
Skipping Q10 (already completed)
Skipping Q11 (already completed)
Skipping Q12 (already completed)
Skipping Q13 (already completed)
Skipping Q14 (already completed)
Skipping Q15 (already completed)
Skipping Q16 (already completed)
Skipping Q17 (already completed)
Skipping Q18 (already completed)

## 7. Review Report

In [7]:
print(f'Report ready at: {report_file}')
if report_file.exists():
    print(f'Total size: {report_file.stat().st_size / 1024:.1f} KB')

    # Preview the first query results inline
    from IPython.display import Markdown, display
    with open(report_file, 'r', encoding='utf-8') as f:
        preview = ''.join(f.readlines()[:60])
    display(Markdown(preview))
else:
    print('No report found.')

Report ready at: /kaggle/working/rag_evaluation_results.md
Total size: 75.3 KB


# FilmDB RAG Evaluation Report
> Corpus: 99k movies | Bi-encoder: BGE-base | Reranker: BGE-reranker-large | LLM: llama-3.3-70b-versatile
---
## Q01: Recommend me a mind-bending movie about dreams and shifting reality.
**Strategy:** `S1_Narrative` | **Time:** 3.7s

### Mode A — Dense Only (cosine similarity)
| Title | Year | IMDB | Votes | Sim Score |
|---|---|---|---|---|
| Dream | 2008 | 6.5 | 4,175 | 0.629 |
| What Dreams May Come | 1998 | 6.9 | 112,493 | 0.629 |
| Altered States | 1980 | 6.9 | 37,342 | 0.624 |
| In Dreams | 1999 | 5.5 | 13,444 | 0.618 |
| The Mandela Effect | 2019 | 5.8 | 5,684 | 0.617 |
| Dream City | 1973 | 6.0 | 263 | 0.615 |
| Passion of Mind | 2000 | 5.5 | 3,791 | 0.613 |
| Dream | 2016 | 5.5 | 351 | 0.612 |
| Poveliteli snov | 2015 | 1.3 | 226 | 0.610 |
| Premonition | 2007 | 5.9 | 80,709 | 0.610 |

### Mode B — Dense + Cross-Encoder Rerank
| Title | Year | IMDB | Votes | Rerank Score |
|---|---|---|---|---|
| Lucia | 2013 | 8.3 | 12,799 | 0.132 |
| Inception | 2010 | 8.8 | 2,473,068 | 0.110 |
| Altered States | 1980 | 6.9 | 37,342 | 0.049 |
| Vanilla Sky | 2001 | 6.9 | 280,276 | 0.037 |
| What Dreams May Come | 1998 | 6.9 | 112,493 | 0.032 |
| In Dreams | 1999 | 5.5 | 13,444 | 0.028 |
| Waking Life | 2001 | 7.7 | 66,105 | 0.022 |
| MindGamers | 2015 | 3.6 | 1,298 | 0.022 |
| Till Sun Rises | 2021 | 4.8 | 337 | 0.021 |
| The Science of Sleep | 2006 | 7.2 | 70,376 | 0.021 |

### Mode C — Full Pipeline (LLM Curated)
### Recommended Films
| Film | Why it fits | Caveats |
| --- | --- | --- |
| **Inception (2010)** | Inception is a mind-bending movie that delves into the concept of shared dreaming, where the lines between reality and dreams are constantly blurred. The film's complex narrative and layered dream sequences make it a perfect fit for the request. With its high-octane action and intellectual depth, it's a thrilling ride that explores the possibilities of dream-sharing. | High-concept sci-fi, intense action sequences |
| **Waking Life (2001)** | Waking Life is an animated film that explores the nature of reality and dreams through a series of philosophical conversations and surreal sequences. The film's dreamlike atmosphere and exploration of the human condition make it a great fit for the request. Its unique animation style and thought-provoking dialogue add to its mind-bending quality. | Animated, philosophical themes, slow-paced |
| **Lucia (2013)** | Lucia is a sci-fi thriller that explores the concept of lucid dreaming, where the protagonist can control her dreams. The film's use of non-linear narrative and blurred lines between reality and dreams make it a great fit for the request. Its low-budget, indie aesthetic adds to its gritty and realistic tone. | Indian film with English subtitles, some violence |
| **Vanilla Sky (2001)** | Vanilla Sky is a psychological thriller that explores the blurring of reality and dreams through the story of a man who becomes trapped in a world of his own making. The film's complex narrative and use of symbolism make it a mind-bending watch. Its exploration of themes such as identity, mortality, and the power of the human mind add to its depth. | Some graphic content, complex narrative |
| **Altered States (1980)** | Altered States is a sci-fi horror film that explores the concept of sensory deprivation and the blurring of reality and dreams. The film's use of practical effects and its exploration of the human psyche make it a great fit for the request. Its slow-burning tension and eerie atmosphere add to its mind-bending quality. | Older film with some dated special effects, graphic content |

---
## Q02: Recommend me a quiet, slice-of-life story about a family in Tokyo.
**Strategy:** `S1_Narrative` | **Time:** 3.0s

### Mode A — Dense Only (cosine similarity)
| Title | Year | IMDB | Votes | Sim Score |
|---|---|---|---|---|
| Tokyo Story | 1953 | 8.1 | 65,916 | 0.681 |
| Tokyo Family | 2013 | 7.5 | 2,357 | 0.674 |
| Still Walking | 2008 | 7.9 | 17,811 | 0.625 |
| The Taste of Tea | 2004 | 7.6 | 7,835 | 0.625 |
| An Inn in Tokyo | 1935 | 7.4 | 1,820 | 0.619 |
| Tokyo Tower: Mom and Me, and Sometimes D | 2007 | 7.3 | 1,124 | 0.618 |
| Nobody Knows | 2004 | 8.0 | 30,106 | 0.617 |
| A Quiet Life | 1995 | 6.9 | 318 | 0.617 |
| Kabei: Our Mother | 2008 | 7.6 | 1,583 | 0.615 |


## Done!
> Download `rag_evaluation_results.md` and `showcase_results.json` from the Kaggle Output panel.
> The JSON file is used by `06_showcase.ipynb` to render interactive visualizations without a GPU.